In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# --- Sklearn Imports ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer # Import this
from sklearn.pipeline import Pipeline # Import this

In [2]:
data_file = Path('../data/T100_domestic/processed/final_enriched_t100_data.parquet')
df = pd.read_parquet(data_file)

In [3]:
df.columns

Index(['PASSENGERS', 'FREIGHT', 'MAIL', 'DISTANCE', 'UNIQUE_CARRIER',
       'AIRLINE_ID', 'UNIQUE_CARRIER_NAME', 'UNIQUE_CARRIER_ENTITY', 'REGION',
       'CARRIER', 'CARRIER_NAME', 'CARRIER_GROUP', 'CARRIER_GROUP_NEW',
       'ORIGIN_AIRPORT_ID', 'ORIGIN_AIRPORT_SEQ_ID', 'ORIGIN_CITY_MARKET_ID',
       'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR', 'ORIGIN_STATE_FIPS',
       'ORIGIN_STATE_NM', 'ORIGIN_COUNTRY', 'ORIGIN_COUNTRY_NAME',
       'ORIGIN_WAC', 'DEST_AIRPORT_ID', 'DEST_AIRPORT_SEQ_ID',
       'DEST_CITY_MARKET_ID', 'DEST', 'DEST_CITY_NAME', 'DEST_STATE_ABR',
       'DEST_STATE_FIPS', 'DEST_STATE_NM', 'DEST_COUNTRY', 'DEST_COUNTRY_NAME',
       'DEST_WAC', 'YEAR', 'QUARTER', 'MONTH', 'DISTANCE_GROUP', 'CLASS',
       'DATA_SOURCE', 'File_name', 'date', 'origin_airport_name',
       'origin_airport_type', 'origin_country_name', 'origin_continent',
       'origin_iso_country', 'dest_airport_name', 'dest_airport_type',
       'dest_country_name', 'dest_continent', 'dest_iso

In [4]:
# --- 2. Feature Engineering & Preprocessing ---
print("--- Preparing data for modeling ---")

categorical_features = ['ORIGIN', 'DEST', 'UNIQUE_CARRIER', 'origin_continent', 'dest_continent']
numerical_features = ['YEAR', 'MONTH', 'DISTANCE', 'origin_gdp', 'origin_population', 'dest_gdp', 'dest_population', 'U.S. Gulf Coast Kerosene-Type Jet Fuel Spot Price FOB (Dollars per Gallon)']
target_col = 'PASSENGERS'

# Create the modeling DataFrame, dropping rows where any of these features are missing
df_model = df[categorical_features + numerical_features + [target_col]].dropna()

# --- 3. Split Data (Train/Test) ---
X = df_model.drop(target_col, axis=1)
y = df_model[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- 4. Create Preprocessing Pipelines ---
# This is the memory-efficient way to handle mixed data types

# Pipeline for numerical features: just scaling
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Pipeline for categorical features: encode as integers
categorical_transformer = Pipeline(steps=[
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

# --- 5. Combine Pipelines with ColumnTransformer ---
# This applies the correct transformation to the correct columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough' # Keep any columns not listed
)

# --- 6. Create the Full Modeling Pipeline ---
# This chainlinks the preprocessor and the model
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=2))
])


--- Preparing data for modeling ---


In [ ]:

# --- 7. Train the Model ---
print("--- Training Random Forest pipeline... ---")
# Fit the entire pipeline on the raw (unscaled, unencoded) training data
rf_pipeline.fit(X_train, y_train)



--- Training Random Forest pipeline... ---


In [ ]:
# --- 8. Evaluate the Model ---
print("\n--- Evaluating model performance ---")
predictions = rf_pipeline.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"Test Set R-squared: {r2:.2%}")
print(f"Test Set RMSE: {rmse:,.2f} passengers")